In [3]:
# ==============================
# 1. IMPORT
# ==============================
import sqlite3
import pandas as pd

# ==============================
# 2. TẠO DATABASE
# ==============================
conn = sqlite3.connect("student.db")
cursor = conn.cursor()

# ==============================
# 3. TẠO BẢNG
# ==============================
cursor.execute("DROP TABLE IF EXISTS student")
cursor.execute("DROP TABLE IF EXISTS course")

cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

# ==============================
# 4. INSERT DATA
# ==============================
students = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0)
]

courses = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)
cursor.executemany("INSERT INTO course VALUES (?, ?)", courses)

conn.commit()

# ==============================
# 5. JOIN (ĐẦY ĐỦ)
# ==============================

print("\n=== CROSS JOIN ===")
display(pd.read_sql("""
SELECT * FROM student CROSS JOIN course
""", conn))

print("\n=== INNER JOIN ===")
display(pd.read_sql("""
SELECT * FROM student s
INNER JOIN course c ON s.course_id = c.id
""", conn))

print("\n=== LEFT JOIN ===")
display(pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
""", conn))

print("\n=== RIGHT JOIN (GIẢ LẬP) ===")
display(pd.read_sql("""
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn))

print("\n=== FULL OUTER JOIN (GIẢ LẬP) ===")
display(pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn))

# ==============================
# 6. LÀM SẠCH DỮ LIỆU
# ==============================

cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")

cursor.execute("""
UPDATE student
SET course_id = 12
WHERE course_id IS NULL
""")

conn.commit()

# ==============================
# 7. THỐNG KÊ THEO LỚP
# ==============================
print("\n=== THỐNG KÊ THEO LỚP ===")
display(pd.read_sql("""
SELECT class,
       COUNT(*) AS so_sinh_vien,
       ROUND(AVG(score),2) AS diem_tb
FROM student
GROUP BY class
""", conn))

# ==============================
# 8. THỐNG KÊ THEO MÔN
# ==============================
print("\n=== THỐNG KÊ THEO MÔN ===")
display(pd.read_sql("""
SELECT c.course_name,
       COUNT(*) AS so_sinh_vien,
       ROUND(AVG(s.score),2) AS diem_tb
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))

# ==============================
# 9. PHÂN LOẠI
# ==============================
print("\n=== PHÂN LOẠI ===")
display(pd.read_sql("""
SELECT c.course_name,
       ROUND(AVG(s.score),2) AS diem_tb,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))

# ==============================
# 10. XẾP HẠNG
# ==============================

print("\n=== XẾP HẠNG TOÀN BỘ ===")
display(pd.read_sql("""
SELECT *,
       RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn))

print("\n=== XẾP HẠNG THEO LỚP ===")
display(pd.read_sql("""
SELECT *,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student
""", conn))

print("\n=== XẾP HẠNG THEO MÔN ===")
display(pd.read_sql("""
SELECT s.*, c.course_name,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank
FROM student s
JOIN course c ON s.course_id = c.id
""", conn))

# ==============================
# 11. TOP 3
# ==============================

print("\n=== TOP 3 CAO NHẤT ===")
display(pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn))

print("\n=== TOP 3 THẤP NHẤT ===")
display(pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score ASC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn))

# ==============================
# 12. GRADUATION DATE
# ==============================

cursor.execute("ALTER TABLE student ADD COLUMN graduation_date TEXT")

cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' ||
    (SELECT RANK() OVER (ORDER BY score DESC)
     FROM student s2
     WHERE s2.student_id = student.student_id)
|| ' days')
""")

conn.commit()

print("\n=== KẾT QUẢ CUỐI ===")
display(pd.read_sql("SELECT * FROM student", conn))

conn.close()


=== CROSS JOIN ===


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich



=== INNER JOIN ===


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke



=== LEFT JOIN ===


...

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3



=== TOP 3 THẤP NHẤT ===


,student_id,name,class,course_id,score,rank
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,7.0,2
2,9,Duong Huu Phuc,Kinh Te,12,7.2,3



=== KẾT QUẢ CUỐI ===


,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-04 16:48:24
1,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-04 16:48:24
2,3,Pham Van Nam,Toan Tin,12,7.9,2026-04-04 16:48:24
3,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-04 16:48:24
4,9,Duong Huu Phuc,Kinh Te,12,7.2,2026-04-04 16:48:24
5,10,Cao Thi Hanh,May Tinh,12,7.0,2026-04-04 16:48:24
